# Unified Qwen Engine: Fast Multi-GPU Inference (llama.cpp Style)
### Bare-Metal CUDA Acceleration for Kaggle Dual NVIDIA Tesla T4 GPUs (2x 16GB `sm_75`)
### Powered by Pre-Compiled Release Wheels (`v0.2.0`) & Hugging Face Hub (`@areen09`)

Run bare-metal CUDA inference on open-weights Qwen models in just a few lines of Python—just like `llama.cpp`!

* **Zero-Wait Setup**: Installs pre-compiled `qwen-engine` wheels with static CUDA 12.4 in ~2 seconds.
* **Turing Tensor Core GEMV**: W4A16 Group-128 quantization (~796 MB VRAM for 1.5B).
* **Linear Attention Hybrid**: 75% KV cache memory reduction via Gated DeltaNet (Qwen 3.5-4B).
* **Dual-GPU Tensor Parallelism**: Shards large 27B model across `cuda:0` and `cuda:1` via peer-to-peer PCIe memory transfers.

## Step 1: Install Pre-Compiled Qwen Engine (~2 Seconds)

Installs the official release wheel built for Linux (`manylinux_2_28`) and Turing `sm_75`.

In [ ]:
import sys

# Auto-detect Python ABI tag (e.g. cp310, cp311)
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
wheel_url = f"https://github.com/Areen-09/qwen-engine/releases/download/v0.2.0/qwen_engine-0.2.0-{py_tag}-{py_tag}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"

print(f"[Setup] Installing pre-compiled release wheel for {py_tag}...")
!pip install -q --no-cache-dir {wheel_url} huggingface_hub

import qwen_engine
from qwen_engine import Engine
print(f"\n✓ Qwen Engine successfully loaded! Version: {getattr(qwen_engine, '__version__', '0.2.0')}")

## Step 2: Download Model Weights (~5 Seconds)

Downloads the lightweight **Qwen 2.5-1.5B W4A16** model (796 MB) and Tokenizer (3.7 MB) from your Hugging Face Hub repository: [`areen09/qwen-engine-models-qwenbin`](https://huggingface.co/areen09/qwen-engine-models-qwenbin).

In [ ]:
from huggingface_hub import hf_hub_download
import os

REPO_ID = "areen09/qwen-engine-models-qwenbin"

print("[Hub Download] Fetching Qwen 2.5-1.5B W4A16 (796 MB)...")
model_15b_path = hf_hub_download(repo_id=REPO_ID, filename="qwen2.5-1.5b-w4a16.qwenbin")

print("[Hub Download] Fetching Tokenizer (3.7 MB)...")
tokenizer_path = hf_hub_download(repo_id=REPO_ID, filename="tokenizer.bin")

print(f"\n✓ Ready! Model: {model_15b_path} ({os.path.getsize(model_15b_path) / (1024**2):.1f} MB)")

## Step 3: Run Streaming Inference (Tesla T4 `cuda:0`)

Initialize the engine and stream tokens in real-time. Notice how clean and simple it is—just like `llama.cpp`!

In [ ]:
import time
from qwen_engine import Engine

# 1. Initialize engine on GPU 0
engine = Engine(model_path=model_15b_path, tokenizer_path=tokenizer_path, device_id=0)

# 2. Stream generation in real time
prompt = "Explain why the sky is blue and how Rayleigh scattering works in two concise bullet points:"
print(f"Prompt:\n{prompt}\n")
print("Response:")
print("-" * 60)

t0 = time.perf_counter()
first_token_time = None
tokens = 0

for piece in engine.stream(prompt, max_tokens=150, temperature=0.7):
    if first_token_time is None:
        first_token_time = time.perf_counter()
    print(piece, end="", flush=True)
    tokens += 1

t1 = time.perf_counter()
print("\n" + "-" * 60)

ttft_ms = (first_token_time - t0) * 1000.0 if first_token_time else 0.0
decode_tps = (tokens - 1) / (t1 - first_token_time) if first_token_time and (t1 > first_token_time) else 0.0

print(f"\nPerformance Metrics:")
print(f"  TTFT (Time to First Token): {ttft_ms:.1f} ms")
print(f"  Decode Throughput:          {decode_tps:.1f} tokens/sec")
print(f"  Total Generated Tokens:     {tokens}")

## Step 4: Full-Text Generation & VRAM Memory Inspection

You can also run full-text generation in a single call, or use the callable `engine("prompt")` shortcut just like `llama-cpp-python`.

In [ ]:
# 1. Single call full generation
response = engine.generate("Explain how gravity works in one simple sentence.", max_tokens=64)
print(f"Prompt: Explain how gravity works in one simple sentence.\n")
print(f"Response: {response.strip()}\n")

# 2. Inspect discrete VRAM memory footprint on Tesla T4
stats = engine.get_memory_stats()
allocated_mb = stats.total_allocated_bytes / (1024**2)
total_mb = stats.device_total_bytes / (1024**2)
print(f"VRAM Memory Stats (Tesla T4 cuda:0):")
print(f"  Allocated VRAM: {allocated_mb:.1f} MB ({(allocated_mb / total_mb)*100:.1f}% occupancy)")
print(f"  Free VRAM:      {(total_mb - allocated_mb):.1f} MB (leaves ~15 GB free for KV cache & activations)")

## Step 5: (Optional) Hybrid Gated DeltaNet (Qwen 3.5-4B)

Qwen 3.5-4B replaces standard attention in 24 of its 32 layers with linear recurrent state updates ($S_t = S_{t-1} + eta_t (v_t - S_{t-1} k_t) k_t^T$), giving **75% KV cache memory reduction**!

In [ ]:
RUN_4B_HYBRID = False  # Set to True to download and test Qwen 3.5-4B (~2.4 GB)

if RUN_4B_HYBRID:
    print("[Hub Download] Fetching Qwen 3.5-4B W4A16 (~2.4 GB)...")
    m4b_path = hf_hub_download(repo_id=REPO_ID, filename="qwen3.5-4b-w4a16.qwenbin")
    
    engine_4b = Engine(model_path=m4b_path, tokenizer_path=tokenizer_path, device_id=0)
    print("\nStreaming from 32-Layer Hybrid DeltaNet:")
    for piece in engine_4b.stream("Describe the difference between linear recurrence and softmax attention:", max_tokens=100):
        print(piece, end="", flush=True)
    print()
else:
    print("[Info] Set RUN_4B_HYBRID = True above to run Qwen 3.5-4B Hybrid DeltaNet.")

## Step 6: (Optional) Dual Tesla T4 2-Way Tensor Parallelism ($TP = 2$)

For the large 27B model, pass `device_ids=[0, 1]` to automatically shard weights across both Tesla T4 GPUs with peer-to-peer (P2P) AllReduce over PCIe!

In [ ]:
RUN_27B_DUAL_GPU = False  # Set to True to test Dual-GPU 2-Way Tensor Parallelism

if RUN_27B_DUAL_GPU:
    print("[Hub Download] Fetching Qwen 3.8-27B TP=2 Ranks (~7.1 GB each)...")
    rank0_path = hf_hub_download(repo_id=REPO_ID, filename="qwen3.8-27b-tp2-rank0-w4a16.qwenbin")
    rank1_path = hf_hub_download(repo_id=REPO_ID, filename="qwen3.8-27b-tp2-rank1-w4a16.qwenbin")
    
    # Passing device_ids=[0, 1] shards Rank 0 on cuda:0 and Rank 1 on cuda:1
    engine_tp = Engine(model_path=rank0_path, tokenizer_path=tokenizer_path, device_ids=[0, 1])
    print("\nStreaming from Dual Tesla T4s (TP=2 across cuda:0 and cuda:1):")
    for piece in engine_tp.stream("Explain the significance of tensor parallelism in large language models:", max_tokens=100):
        print(piece, end="", flush=True)
    print()
else:
    print("[Info] Set RUN_27B_DUAL_GPU = True above to run Qwen 3.8-27B sharded across both GPUs.")